In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/krupalpatel07/blackrock/BLACKROCK.csv


In [2]:

# =====================================================
# 1. IMPORTS
# =====================================================

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "iframe"

import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans

plt.style.use("dark_background")

# =====================================================
# 2. LOAD DATA
# =====================================================

file_path = "/kaggle/input/datasets/krupalpatel07/blackrock/BLACKROCK.csv"

df = pd.read_csv(file_path)

df.columns = [c.lower() for c in df.columns]

df["date"] = pd.to_datetime(df["date"])

df = df.sort_values("date")

df.set_index("date", inplace=True)

# =====================================================
# 3. BLACKROCK HEADER
# =====================================================

from IPython.display import HTML, display

def aladdin_header(title):

    display(HTML(f"""
    <div style="
    background:linear-gradient(
    135deg,
    #000000,
    #111111,
    #D4AF37
    );
    padding:24px;
    border-radius:20px;
    margin-top:25px;
    box-shadow:0px 0px 30px rgba(212,175,55,0.4);
    ">
        <h1 style="
        color:white;
        text-align:center;
        font-size:38px;
        letter-spacing:2px;">
        {title}
        </h1>
    </div>
    """))

aladdin_header("🏛 Institutional Command Center")

In [3]:
aladdin_header("📊 Executive Dashboard")

df["returns"] = df["close"].pct_change()

total_return = (
    (df["close"].iloc[-1] /
     df["close"].iloc[0]) - 1
) * 100

volatility = (
    df["returns"].std()
    * np.sqrt(252)
    * 100
)

sharpe = (
    df["returns"].mean()
    /
    df["returns"].std()
) * np.sqrt(252)

cards = pd.DataFrame({

    "Metric":[
        "Total Return %",
        "Volatility %",
        "Sharpe Ratio"
    ],

    "Value":[
        round(total_return,2),
        round(volatility,2),
        round(sharpe,2)
    ]

})

fig = px.bar(
    cards,
    x="Metric",
    y="Value",
    text="Value",
    title="Institutional KPI Dashboard"
)

fig.show()

/tmp/ipykernel_58/3857574763.py:3: FutureWarning:

The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.



In [4]:
aladdin_header("🧠 Alpha Factor Laboratory")

df["mom_21"] = df["close"].pct_change(21)

df["mom_63"] = df["close"].pct_change(63)

df["mom_126"] = df["close"].pct_change(126)

df["quality"] = (

    (
        df["close"].rolling(50).std()
    ).rank(pct=True)

)

df["value_proxy"] = (

    df["close"]

    /

    df["close"].rolling(200).mean()

)

df["alpha_score"] = (

    df["mom_21"] * 0.3 +

    df["mom_63"] * 0.3 +

    df["mom_126"] * 0.2 +

    df["quality"] * 0.1 +

    df["value_proxy"] * 0.1

)

fig = px.line(
    df,
    y="alpha_score",
    title="Composite Alpha Score"
)

fig.show()

/tmp/ipykernel_58/3313656063.py:3: FutureWarning:

The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.

/tmp/ipykernel_58/3313656063.py:5: FutureWarning:

The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.

/tmp/ipykernel_58/3313656063.py:7: FutureWarning:

The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.



In [5]:
aladdin_header("🔥 Institutional Regime Engine")

vol = df["returns"].rolling(20).std()

mom = df["close"].pct_change(20)

conditions = [

    (mom > 0) & (vol < vol.median()),

    (mom > 0) & (vol > vol.median()),

    (mom < 0) & (vol < vol.median()),

    (mom < 0) & (vol > vol.median())

]

choices = [

    "Expansion",

    "Accumulation",

    "Recovery",

    "Panic"

]

df["regime"] = np.select(
    conditions,
    choices,
    default="Neutral"
)

fig = px.scatter(
    df,
    x=df.index,
    y="close",
    color="regime",
    title="Market Regime Rotation"
)

fig.show()

/tmp/ipykernel_58/362158277.py:5: FutureWarning:

The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.



In [6]:
aladdin_header("🌌 Hidden Alpha Detector")

df["hidden_alpha"] = (

    np.abs(df["returns"])

    *

    np.log1p(df["volume"])

    /

    (vol + 0.0001)

)

fig = px.area(
    df,
    y="hidden_alpha",
    title="Hidden Alpha Bursts"
)

fig.show()

In [7]:
aladdin_header("⚖️ Smart Beta Engine")

df["momentum_beta"] = (
    df["mom_63"]
)

df["volatility_beta"] = (
    -vol
)

df["liquidity_beta"] = (
    np.log1p(df["volume"])
)

scaler = MinMaxScaler()

beta_data = scaler.fit_transform(

    df[
        [
            "momentum_beta",
            "volatility_beta",
            "liquidity_beta"
        ]
    ].fillna(0)

)

df["smart_beta"] = beta_data.mean(axis=1)

fig = px.line(
    df,
    y="smart_beta",
    title="Smart Beta Composite"
)

fig.show()

In [8]:
aladdin_header("🤖 Institutional Market States")

cluster_data = df[

    [
        "alpha_score",
        "smart_beta",
        "hidden_alpha"
    ]

].fillna(0)

model = KMeans(
    n_clusters=5,
    random_state=42
)

df["state"] = model.fit_predict(
    cluster_data
)

fig = px.scatter(
    df,
    x=df.index,
    y="close",
    color=df["state"].astype(str),
    title="Institutional Market States"
)

fig.show()

In [9]:
aladdin_header("🎯 BlackRock Alpha Engine")

signal = (

    (df["alpha_score"] > df["alpha_score"].rolling(50).mean())

    &

    (df["smart_beta"] >
     df["smart_beta"].rolling(50).mean())

)

df["signal"] = signal.astype(int)

fig = go.Figure()

fig.add_trace(

    go.Scatter(
        x=df.index,
        y=df["close"],
        name="Price"
    )
)

fig.add_trace(

    go.Scatter(
        x=df.index[df["signal"]==1],
        y=df["close"][df["signal"]==1],
        mode="markers",
        name="Alpha Zone"
    )
)

fig.show()

In [10]:
aladdin_header("📌 Aladdin Intelligence")

print("""

1. Factor Investing identifies persistent drivers.

2. Smart Beta enhances traditional momentum.

3. Hidden Alpha reveals institutional activity.

4. Regime Rotation improves market timing.

5. Multi-factor models create robust signals.

6. Institutional thinking beats indicator stacking.

""")



1. Factor Investing identifies persistent drivers.

2. Smart Beta enhances traditional momentum.

3. Hidden Alpha reveals institutional activity.

4. Regime Rotation improves market timing.

5. Multi-factor models create robust signals.

6. Institutional thinking beats indicator stacking.


